In [3]:
from importlib.metadata import version

pkgs = [
    "numpy",       # PyTorch & TensorFlow dependency
    "matplotlib",  # Plotting library
    "tiktoken",    # Tokenizer
    "torch",       # Deep learning library
    # "tqdm",        # Progress bar
    # "tensorflow",  # For OpenAI's pretrained weights
]
for p in pkgs:
    print(f"{p} version: {version(p)}")

numpy version: 2.4.2
matplotlib version: 3.10.8
tiktoken version: 0.12.0
torch version: 2.10.0


Setting Device

In [7]:
import torch
import torch.nn as nn

# Check if CUDA is available and set device accordingly
device = torch.device("cpu")
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
print(f"Using device: {device}")


Using device: mps


Tensor
-  A 32-bit floating point number offers sufficient precision for most deep learning tasks, while consuming less memory and computational resources than a 64-bit floating point number. Moreover, GPU architectures are optimized for 32-bit computations, and using this data type can significantly speed up model training and inference.

In [12]:
# create a 0D tensor (scalar) from a Python integer
tensor0d = torch.tensor(1)
print("--- 0D Tensor ---")
print(tensor0d)
print(tensor0d.shape)  # Output: torch.Size([])

# create a 1D tensor (vector) from a Python list
tensor1d = torch.tensor([1, 2, 3])
print("--- 1D Tensor ---")
print(tensor1d)
print(tensor1d.shape)  # Output: torch.Size([3])
print(tensor1d.dtype)

# create a 2D tensor from a nested Python list
tensor2d = torch.tensor([[1, 2], [3, 4]])
print("--- 2D Tensor ---")
print(tensor2d)
print(tensor2d.shape)  # Output: torch.Size([2, 2])
# create a 3D tensor from a nested Python list
tensor3d = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]])
print("--- 3D Tensor ---")
print(tensor3d)
print(tensor3d.shape)  # Output: torch.Size([2, 2, 2])

floatvec = torch.tensor([1.0, 2.0, 3.0])
print("--- Float Vector ---")
print(floatvec.dtype)


--- 0D Tensor ---
tensor(1)
torch.Size([])
--- 1D Tensor ---
tensor([1, 2, 3])
torch.Size([3])
torch.int64
--- 2D Tensor ---
tensor([[1, 2],
        [3, 4]])
torch.Size([2, 2])
--- 3D Tensor ---
tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])
torch.Size([2, 2, 2])
--- Float Vector ---
torch.float32


Tensor Operation

In [19]:
tensor2d = torch.tensor([[1, 2, 3],
                         [4, 5, 6]])

# reshape 
reshaped_tensor = tensor2d.reshape(3, 2)
print("--- Reshaped Tensor ---")
print(tensor2d)
print(reshaped_tensor)

# view
view_tensor = tensor2d.view(3, 2)
print("--- View Tensor ---")
print(tensor2d)
print(view_tensor)

# transpose
transposed_tensor = tensor2d.T
print("--- Transposed Tensor ---")
print(tensor2d)
print(transposed_tensor)


# multiplicaiton 
print("--- Multiplication ---")
print(tensor2d.matmul(transposed_tensor))
print(tensor2d @ transposed_tensor)



--- Reshaped Tensor ---
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
--- View Tensor ---
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2],
        [3, 4],
        [5, 6]])
--- Transposed Tensor ---
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 4],
        [2, 5],
        [3, 6]])
--- Multiplication ---
tensor([[14, 32],
        [32, 77]])
tensor([[14, 32],
        [32, 77]])


Computational Grapth


<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_08.webp" width=500px>


In [20]:
import torch.nn.functional as F

y = torch.tensor([1.0])  # true label
x1 = torch.tensor([1.1]) # input feature
w1 = torch.tensor([2.2]) # weight parameter
b = torch.tensor([0.0])  # bias unit

z = x1 * w1 + b          # net input
a = torch.sigmoid(z)     # activation & output

loss = F.binary_cross_entropy(a, y)
print(loss)


tensor(0.0852)


Autograd

In [ ]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)




(tensor([-0.0898]),)
(tensor([-0.0817]),)


Above, we have been using the grad function “manually,” which can be useful for experimentation, debugging, and demonstrating concepts. But in practice, PyTorch provides even more high-level tools to automate this process. For instance, we can call .backward on the loss, and PyTorch will compute the gradients of all the leaf nodes in the graph, which will be stored via the tensors’ .grad attributes:

In [24]:
loss.backward()
print(w1.grad)
print(b.grad)

tensor([-0.0898])
tensor([-0.0817])


NN


<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_09.webp" width=500px>

In [35]:
import torch.nn as nn

torch.manual_seed(123)

class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(

            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            # output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        '''
        Note that we used the Sequential class when we implemented the NeuralNetwork class. 
        Using Sequential is not required, but it can make our life easier if we have a series of layers that we want to execute in a specific order, as is the case here. 
        This way, after instantiating self.layers = Sequential(...) in the __init__ constructor, we just have to call the self.layers instead of calling each layer individually in the NeuralNetwork’s forward method.
        '''
        logits = self.layers(x)
        return logits

# Create NN and Inspect the parameters 
model = NeuralNetwork(50, 3)
print(model)

sum = 0
for p in model.parameters():
    if p.requires_grad:
        print(p.shape)
        print(p.numel())
        sum += p.numel()
print(f"Total number of parameters: {sum}")

print(model.layers[0].weight.shape)  # Output: torch.Size([30, 50])
print(model.layers[0].weight)    # Output: torch.Size([30])

print(model.layers[0].bias.shape)    # Output: torch.Size([30]
print(model.layers[0].bias)    # Output: torch.Size([30])







NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)
torch.Size([30, 50])
1500
torch.Size([30])
30
torch.Size([20, 30])
600
torch.Size([20])
20
torch.Size([3, 20])
60
torch.Size([3])
3
Total number of parameters: 2213
torch.Size([30, 50])
Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)
torch.Size([30])
Parameter containing:
tensor([-0.1250,  0.0513,  0.0366,  

In particular, grad_fn=<AddmmBackward0> means that the tensor we are inspecting was created via a matrix multiplication and addition operation. PyTorch will use this information when it computes gradients during backpropagation. The <AddmmBackward0> part of grad_fn=<AddmmBackward0> specifies the operation that was performed. In this case, it is an Addmm operation. Addmm stands for matrix multiplication (mm) followed by an addition (Add).

In [36]:
# Forward pass with random input
torch.manual_seed(123)
x = torch.randn(1, 50)  # Random input tensor with shape (1
out = model(x)
print(f"out={out}")

out=tensor([[-0.1080,  0.0924,  0.0775]], grad_fn=<AddmmBackward0>)


If we just want to use a network without training or backpropagation, for example, if we use it for prediction after training, constructing this computational graph for backpropagation can be wasteful as it performs unnecessary computations and consumes additional memory. So, when we use a model for inference (for instance, making predictions) rather than training, it is a best practice to use the torch.no_grad() context manager, as shown below. This tells PyTorch that it doesn’t need to keep track of the gradients, which can result in significant savings in memory and computation.

In [37]:
with torch.no_grad():
    out = model(x)
print(f"out={out}") 


out=tensor([[-0.1080,  0.0924,  0.0775]])


In [39]:
with torch.no_grad():
    out = torch.softmax(model(x), dim=1)
print(out)


tensor([[0.2919, 0.3567, 0.3514]])



<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_10.webp" width=500px>

In [ ]:
# train data
X_train = torch.tensor([
    [-1.2, 3.1],
    [-0.9, 2.9],
    [-0.5, 2.6],
    [2.3, -1.1],
    [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])


# test data
X_test = torch.tensor([
    [-0.8, 2.8],
    [2.6, -1.6],
])

y_test = torch.tensor([0, 1])


# data sets
from torch.utils.data import Dataset
class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
        return self.labels.shape[0]

train_ds = ToyDataset(X_train, y_train)
test_ds = ToyDataset(X_test, y_test)


# data loader
from torch.utils.data import DataLoader

torch.manual_seed(123)

train_loader = DataLoader(
    dataset=train_ds,
    batch_size=2,
    shuffle=True,
    # When num_workers is set to 0, the data loading will be done in the main process and not in separate worker processes.
    num_workers=0,
    # In practice, having a substantially smaller batch as the last batch in a training epoch can disturb the convergence during training. 
    # To prevent this, it’s recommended to set drop_last=True, which will drop the last batch in each epoch
    drop_last=True
)

test_loader = DataLoader(
    dataset=test_ds,
    batch_size=2,
    shuffle=False,
    num_workers=0
)

In [46]:
for idx, (x, y) in enumerate(train_loader):
    print(f"Batch {idx}:")
    print(f"x={x}")
    print(f"y={y}")

Batch 0:
x=tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]])
y=tensor([1, 0])
Batch 1:
x=tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]])
y=tensor([0, 0])


When num_workers is set to 0, the data loading will be done in the main process and not in separate worker processes. This might seem unproblematic, but it can lead to significant slowdowns during model training when we train larger networks on a GPU. This is because instead of focusing solely on the processing of the deep learning model, the CPU must also take time to load and preprocess the data. As a result, the GPU can sit idle while waiting for the CPU to finish these tasks. In contrast, when num_workers is set to a number greater than zero, multiple worker processes are launched to load data in parallel, freeing the main process to focus on training your model and better utilizing your system’s resources.

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_11.webp" width=800px>


Training Loop


In practice, we often use a third dataset, a so-called validation dataset, to find the optimal hyperparameter settings. A validation dataset is similar to a test set. However, while we only want to use a test set precisely once to avoid biasing the evaluation, we usually use the validation set multiple times to tweak the model settings.

In [ ]:
import torch.nn.functional as F


torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

# stochastic gradient descent (SGD) optimizer with a learning rate (lr) of 0.5.
# lr is tunable hyperparameter that controls the step size at each iteration while moving toward a minimum of a loss function.
# Ideally, we want to choose a learning rate such that the loss converges after a certain number of epochs – 
# the number of epochs is another hyperparameter to choose.
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    # model.train() and model.eval() put the model into a training and an evaluation mode
    # This is necessary for components that behave differently during training and inference, such as dropout or batch normalization layers.
    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        logits = model(features)

        loss = F.cross_entropy(logits, labels) # Loss function

        # reset grad make sure no accumulation from previous batches
        optimizer.zero_grad()

        #  will calculate the gradients in the computation graph that PyTorch constructed in the background. 
        loss.backward()

        # method will use the gradients to update the model parameters to minimize the loss.
        # In the case of the SGD optimizer, this means multiplying the gradients with the learning rate and adding the scaled negative gradient to the parameters.
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")


Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


In [55]:
model.eval()

with torch.no_grad():
    outputs = model(X_train)

probas = torch.softmax(outputs, dim=1)
torch.set_printoptions(sci_mode=False)
print(probas)

# We can convert these values into class labels predictions using PyTorch’s argmax function, which returns the index position of the highest value in each row if we set dim=1 (setting dim=0 would return the highest value in each column, instead):

predictions = torch.argmax(probas, dim=1)
print(predictions)

# check accuracy
correct = (predictions == y_train)
print(correct)
torch.sum(correct)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])
tensor([0, 0, 0, 1, 1])
tensor([True, True, True, True, True])


tensor(5)

In [57]:
def compute_accuracy(model, dataloader):

    model = model.eval()
    correct = 0.0
    total_examples = 0

    for idx, (features, labels) in enumerate(dataloader):

        with torch.no_grad():
            logits = model(features)

        predictions = torch.argmax(logits, dim=1)
        compare = labels == predictions
        correct += torch.sum(compare)
        total_examples += len(compare)

    return (correct / total_examples).item()

print(f"Train accuracy: {compute_accuracy(model, train_loader):.2f}")
print(f"Test accuracy: {compute_accuracy(model, test_loader):.2f}")

Train accuracy: 1.00
Test accuracy: 1.00


Saving and Loading Models

In [58]:
torch.save(model.state_dict(), "model.pth")
model = NeuralNetwork(2, 2) # needs to match the original model exactly
model.load_state_dict(torch.load("model.pth", weights_only=True))


<All keys matched successfully>

GPU Computing


In [ ]:
print(torch.cuda.is_available())
print(torch.backends.mps.is_available())

tensor_1 = torch.tensor([1., 2., 3.])
tensor_2 = torch.tensor([4., 5., 6.])
print(tensor_1 + tensor_2) #tensor([5., 7., 9.])

# If your machine hosts multiple GPUs, you have the option to specify which GPU you’d like to transfer the tensors to. 
# You can do this by indicating the device ID in the transfer command. For instance, you can use .to("cuda:0"), .to("cuda:1"), and so on.

tensor_1 = tensor_1.to("mps")
tensor_2 = tensor_2.to("mps")
print(tensor_1 + tensor_2) #tensor([5., 7., 9.], device='mps:0')


False
True
tensor([5., 7., 9.])
tensor([5., 7., 9.], device='mps:0')


In [ ]:
# single GPU training 

torch.manual_seed(123)
model = NeuralNetwork(num_inputs=2, num_outputs=2)

# New: Define a device variable that defaults to a GPU.
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu") to device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
device = torch.device("mps")
# New: Transfer the model onto the GPU.
model.to(device)

optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    for batch_idx, (features, labels) in enumerate(train_loader):

        # New: Transfer the data onto the GPU.
        features, labels = features.to(device), labels.to(device)    #C
        logits = model(features)
        loss = F.cross_entropy(logits, labels) # Loss function

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        ### LOGGING
        print(f"Epoch: {epoch+1:03d}/{num_epochs:03d}"
              f" | Batch {batch_idx:03d}/{len(train_loader):03d}"
              f" | Train/Val Loss: {loss:.2f}")


Epoch: 001/003 | Batch 000/002 | Train/Val Loss: 0.75
Epoch: 001/003 | Batch 001/002 | Train/Val Loss: 0.65
Epoch: 002/003 | Batch 000/002 | Train/Val Loss: 0.44
Epoch: 002/003 | Batch 001/002 | Train/Val Loss: 0.13
Epoch: 003/003 | Batch 000/002 | Train/Val Loss: 0.03
Epoch: 003/003 | Batch 001/002 | Train/Val Loss: 0.00


Muti GPU training 

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_12.webp" width=800>

DistributedDataParallel (DDP) strategy. DDP enables parallelism by splitting the input data across the available devices and processing these data subsets simultaneously.

PyTorch launches a separate process on each GPU, and each process receives and keeps a copy of the model – these copies will be synchronized during training.Each of the two GPUs will receive a copy of the model. Then, in every training iteration, each model will receive a minibatch (or just batch) from the data loader. We can use a DistributedSampler to ensure that each GPU will receive a different, non-overlapping batch when using DDP.

Since each model copy will see a different sample of the training data, the model copies will return different logits as outputs and compute different gradients during the backward pass. These gradients are then averaged and synchronized during training to update the models. This way, we ensure that the models don’t diverge,

<img src="https://sebastianraschka.com/images/teaching/pytorch-1h/figure_13.webp" width=800>

The benefit of using DDP is the enhanced speed it offers for processing the dataset compared to a single GPU. Barring a minor communication overhead between devices that comes with DDP use, it can theoretically process a training epoch in half the time with two GPUs compared to just one. The time efficiency scales up with the number of GPUs, allowing us to process an epoch eight times faster if we have eight GPU. and so on
